
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# **NumPy Array Programming Guide** - Solutions to the practice exercises

In [ ]:
import numpy as np

## **Exercise 1**

Define the `relu` function that applies the ReLU (Rectified Linear Unit) operation elementwise to an array of arbitrary shape!

`ReLU(z) = max(0, z)`

Example:
```
a = np.array([[1., -3.], [0.5, -0.5], [0., 2.]])
relu(a) -> ndarray([[1., 0.], [0.5, 0.], [0., 2.]])
```


### Hints

**Possible approaches:**
- `np.maximum`. **Difficulty:** 1/5
- `np.where` (with 3 parameters). **Difficulty:** 1/5
- Advanced indexing with a condition. **Difficulty:** 1/5


### Solutions

**Solution 1**

In [ ]:
def relu(a):
  return np.maximum(a, 0)

**Solution 2**

In [ ]:
def relu(a):
  return np.where(a < 0, 0, a)

**Solution 3**

In [ ]:
def relu(a):
  a = a.copy()   # to avoid modifying the input array
  a[a < 0] = 0
  return a

**Tests**

In [ ]:
# Tests
a = np.array([[1., -3.], [0.5, -0.5], [0., 2.]])
a1 = relu(a)
assert np.array_equal(a1, np.array([[1., 0.], [0.5, 0.], [0., 2.]]))
print("Tests were successful.")

## **Exercise 2**

Define the `one_hot_1d` function that converts category labels (indices) into one-hot encoded label vectors!

The `one_hot_1d(arr, n)` function takes a 1D array `arr` of category labels (indices) and converts each of the labels to their one-hot encoded form. The number of categories is given by `n`. The output is a 2D floating-point (`float`) array, where the slice `[i, :]` is the one-hot encoding of the i-th element of the input array.

<font color='gray'>_In one-hot encoding, a category label `k` (a non-negative integer) is converted into a vector of length `n`, where `n` is the number of categories. Every element of the vector is 0, except the k-th element, which is 1. One-hot encoding is commonly used in multi-class classification. For example, we might want to classify photos into 3 categories: dog, cat, parrot. Each photo belongs to exactly one of these categories, so we can assign labels such as 0 for dog, 1 for cat, 2 for parrot. In machine learning, we typically do not predict the label directly; instead, we predict a probability vector giving the probability that the photo shows a dog, a cat, or a parrot. One-hot encoding is used to construct the ground-truth label vectors used for training. For instance, if the image contains a parrot, the corresponding true label vector is `[0., 0., 1.]`, meaning 0 probability for dog and cat, and 1 probability for parrot. The one-hot encoding is also used frequently in natural language processing. Each token in a vocabulary of size `n` can be represented as a one-hot vector of length `n`._</font>

Example:
```
arr = np.array([0, 2, 2, 1])
one_hot_1d(arr, 3) -> ndarray([[1., 0., 0.], [0., 0., 1.], [0., 0., 1.], [0., 1., 0.]])
```


### Hints

**Possible approaches:**
- Initialize the output array with zeros, then write the ones into the appropriate positions using advanced indexing (`np.arange`). **Difficulty:** 2/5
- The rows of an identity matrix (`np.eye`) are the possible one-hot vectors. Use advanced indexing to select the required rows and return the result as a 2D array. **Difficulty:** 2/5
- Use a "multiplication-table" style broadcast to test equality between the labels and all possible labels. **Difficulty:** 2/5
- Each one-hot vector can be obtained from `[0, 1, 0]` by repeating elements the appropriate number of times (possibly zero) using `np.repeat`. If vectorized properly, this yields the desired one-hot array. **Difficulty:** 3/5


### Solutions

**Solution 1**

In [ ]:
def one_hot_1d(arr, n):
  onehots = np.zeros((arr.shape[0], n), dtype=np.float32)
  onehots[np.arange(arr.shape[0]), arr] = 1.
  return onehots

**Solution 2**

In [ ]:
def one_hot_1d(arr, n):
  return np.eye(n)[arr,:]

**Solution 3**

In [ ]:
def one_hot_1d(arr, n):
  return (arr[:,None] == np.arange(n)).astype(np.float32)

**Solution 4**

In [ ]:
def one_hot_1d(arr, n):
  onehots = np.broadcast_to([[0.,1.,0.]], (arr.shape[0], 3))   # (1, 3) -> (n_samples, 3)
  reps = np.stack([arr, np.ones_like(arr), n-arr-1], axis=1).reshape(-1)
  return np.repeat(onehots, reps).reshape((arr.shape[0], n))

**Tests**

In [ ]:
# Tests
arr = np.array([1, 2, 2, 1])
arr1 = one_hot_1d(arr, 3)
assert np.issubdtype(arr1.dtype, np.floating)   # check if result array is of floating point data type (e.g., np.float32)
assert np.allclose(arr1, np.array([[0., 1., 0.], [0., 0., 1.], [0., 0., 1.], [0., 1., 0.]]), atol=0.01)
arr2 = one_hot_1d(arr, 4)
assert np.issubdtype(arr1.dtype, np.floating)   # check if result array is of floating point data type (e.g., np.float32)
assert np.allclose(arr2, np.array([[0., 1., 0., 0.], [0., 0., 1., 0.], [0., 0., 1., 0.], [0., 1., 0., 0.]]), atol=0.01)
print("Tests were successful.")

## **Exercise 3**

Define the `category_label` function that, given multiple probability vectors at once, returns the most likely category index for each vector!

The `category_label(probs_arr)` function takes a 2D array of probability vectors. For each probability vector it returns the index of the most likely category (that is, the index of the maximum element). The result is returned as a 1D array.

<font color='gray'>_In multi-class classification, each sample belongs to one of `k` categories. When using a neural network, we typically predict a length-`k` vector of probabilities that represent the probability of the sample belonging to each category. If we want to turn these vectors into discrete predicted category labels, it is enough to return the index of the maximum element in each vector. When `category_label` is given the output of `one_hot_1d`, it reconstructs the original input labels._</font>

One well-known NumPy function solves this problem directly. Try to also provide a solution where you do not use that function.

Example:
```
arr = np.array([[1., 0., 0.], [0., 0., 1.], [0., 0., 1.], [0., 1., 0.]])
category_label(arr) -> ndarray([0, 2, 2, 1])
```


### Hints

**Possible approaches:**
- `np.argmax`. **Difficulty:** 1/5
- Without `np.argmax`: find the coordinates (`np.where`) of elements that are equal to the maximum of their own row. Then write these indices into an empty result array using advanced indexing in such a way that exactly one element per row is written. **Difficulty:** 3/5


### Solutions

**Solution 1**

In [ ]:
def category_label(probs_arr):
  return np.argmax(probs_arr, axis=1)

**Solution 2**

In [ ]:
def category_label(probs_arr):
  onehot_mask = probs_arr == np.amax(probs_arr, axis=1)[:,None]   # several 1s can be present in each vector (in case of ties)
  one_idxs0, one_idxs1 = np.where(onehot_mask)
  cat_labels = np.empty((probs_arr.shape[0],), dtype=np.int64)
  cat_labels[one_idxs0] = one_idxs1              # in case of duplicates along axis#1, only one of them will be written to cat_labels
  return cat_labels


**Tests**

In [ ]:
# Tests
arr = np.array([[0., 1., 0.], [0., 0., 1.], [0., 0., 1.], [0., 1., 0.]])
r = category_label(arr)
assert np.allclose(r, np.array([1, 2, 2, 1]), atol=0.01)
arr = np.array([[0.5, 0.3, 0.2], [0.09, 0.9, 0.01]])
r = category_label(arr)
assert np.allclose(r, np.array([0., 1.]), atol=0.01)
arr = np.array([[0.4, 0.2, 0.4], [0., 0.5, 0.5]])
r = category_label(arr)
assert r.shape == (2,)
assert (r[0] in [0,2]) and (r[1] in [1,2])
print("Tests were successful.")

## **Exercise 4**

Solve the following tasks related to min-max scaling!

**A)** Define the `minmax_rescale(arr, a, b)` function, which linearly rescales the array `arr` into the interval between `a` and `b` such that the smallest element becomes `a` and the largest element becomes `b`.

Example:
```
arr = np.array([[2., 1.], [-2., 0.]])
minmax_rescale(arr, -1., 1.) -> ndarray([[1., 0.5], [-1., 0.]])
```

**B)** Define the `minmax_rescale_per_var(arr, a, b)` function, which independently rescales the slices of the 2D array `arr` taken along axis #1 (the second axis) into the interval between `a` and `b`, such that within each slice the minimum becomes `a` and the maximum becomes `b`.
That is, for the result array `r`, every slice `r[i, :]` (i = 0, 1, ...) will have minimum `a` and maximum `b`.

Example:
```
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
minmax_rescale_per_var(arr, -1., 1.) -> ndarray([[1., 0., -1.], [-1., 0.6, 1.]])
```

**C)** Define the `minmax_rescale_along_axis(arr, a, b, axis)` function, which independently rescales the slices of the 2D array `arr` taken along the axis specified by `axis` into the interval between `a` and `b`, such that within each slice the minimum becomes `a` and the maximum becomes `b`.
That is, this should match `minmax_rescale_per_var` when `axis=1`. When `axis=0`, rescale the slices taken along the other axis. Try to handle both `axis=0` and `axis=1` in a single vectorized implementation, not with an `if` branch.

Example:
```
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
minmax_rescale_along_axis(arr, -1., 1., axis=0) -> ndarray([[1., 1., -1.], [-1., -1., 1.]])
minmax_rescale_along_axis(arr, -1., 1., axis=1) -> ndarray([[1., 0., -1.], [-1., 0.6, 1.]])
```


### Hints

**Possible approaches:**

**Part A**
- Use the min-max scaling formula (`np.amin`, `np.amax`). **Difficulty:** 1/5

**Part B**
- Compute per-slice min/max (`np.amin`, `np.amax` with the `axis` parameter), then apply the min-max scaling formula with appropriate broadcasting. **Difficulty:** 2/5

**Part C**
- Use `keepdims=True`. **Difficulty:** 2/5


### Solutions

**Part A, solution 1**

In [ ]:
def minmax_rescale(arr, a, b):
  arrmin = np.amin(arr)       # scalar
  arrmax = np.amax(arr)       # scalar
  assert arrmax > arrmin, "There must be at least two different elements in the array."
  return a + ((arr-arrmin)*(b-a))/(arrmax-arrmin)

**Part B, solution 1**

In [ ]:
def minmax_rescale_per_var(arr, a, b):
  arrmin = np.amin(arr, axis=1)       # (arr.shape[0],)
  arrmax = np.amax(arr, axis=1)       # (arr.shape[0],)
  assert np.all(arrmax > arrmin), "There must be at least two different elements in each slice."
  return a + ((arr-arrmin[:,None])*(b-a))/((arrmax-arrmin)[:,None])

**Part C, solution 1**

In [ ]:
def minmax_rescale_along_axis(arr, a, b, axis):
  assert axis in [0, 1]
  arrmin = np.amin(arr, axis=axis, keepdims=True)       # (1, arr.shape[1]) if axis == 0 OR (arr.shape[0], 1) if axis == 1
  arrmax = np.amax(arr, axis=axis, keepdims=True)       # (1, arr.shape[1]) if axis == 0 OR (arr.shape[0], 1) if axis == 1
  assert np.all(arrmax > arrmin), "There must be at least two different elements in each slice."
  return a + ((arr-arrmin)*(b-a))/((arrmax-arrmin))     # automatic broadcasting

**Tests**

In [ ]:
# Tests
if 'minmax_rescale' in locals():
  arr = np.array([[0.5, 1.2], [2.2, 1.0], [1.3, 0.9]])
  arr1 = minmax_rescale(arr, -2., 3.)
  assert np.allclose(arr1, np.array([[-2., 0.0588], [3., -0.529], [0.352, -0.823]]), atol=0.01)
  print("Tests were successful for: 'minmax_rescale'.")
else:
  print("'minmax_rescale' not found, tests skipped.")

if 'minmax_rescale_per_var' in locals():
  arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
  arr1 = minmax_rescale_per_var(arr, -1., 1.)
  assert np.allclose(arr1, np.array([[1., 0., -1.], [-1., .6, 1.]]), atol=0.01)
  print("Tests were successful for: 'minmax_rescale_per_var'.")
else:
  print("'minmax_rescale_per_var' not found, tests skipped.")


if 'minmax_rescale_along_axis' in locals():
  arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
  arr1 = minmax_rescale_along_axis(arr, -1., 1., axis=0)
  assert np.allclose(arr1, np.array([[1., 1., -1.], [-1., -1., 1.]]), atol=0.01)
  arr1 = minmax_rescale_along_axis(arr, -1., 1., axis=1)
  assert np.allclose(arr1, np.array([[1., 0., -1.], [-1., .6, 1.]]), atol=0.01)
  print("Tests were successful for: 'minmax_rescale_along_axis'.")
else:
  print("'minmax_rescale_along_axis' not found, tests skipped.")


## **Exercise 5**

Define the `standardize_along_axis(arr, axis)` function, which independently standardizes the slices of the 2D array `arr` taken along the axis specified by `axis`. For each slice, subtract the slice mean from every element and divide by the slice standard deviation.
That is, for example when `axis=1`, in the result array `r` every slice `r[i, :]` (i = 0, 1, ...) will have mean 0 and standard deviation 1.

Example:
```
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
standardize_along_axis(arr, axis=1) -> ndarray([[1.224, 0., -1.224], [-1.388, 0.462, 0.925]])
```


### Hints

**Possible approaches:**
- Similar to `minmax_rescale_along_axis`. Use `np.mean` and `np.std`. **Difficulty:** 2/5


### Solutions

**Solution 1**

In [ ]:
def standardize_along_axis(arr, axis):
  assert axis in [0, 1]
  EPS = 1e-8                 # to avoid division by zero
  arrmean = np.mean(arr, axis=axis, keepdims=True)       # (1, arr.shape[1]) if axis == 0 OR (arr.shape[0], 1) if axis == 1
  arrstd = np.std(arr, axis=axis, keepdims=True)         # (1, arr.shape[1]) if axis == 0 OR (arr.shape[0], 1) if axis == 1
  return (arr - arrmean)/(arrstd + EPS)     # automatic broadcasting

**Tests**

In [ ]:
# Tests
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
arr1 = standardize_along_axis(arr, axis=0)
assert np.allclose(arr1, np.array([[1., 1., -1.], [-1., -1., 1.]]), atol=0.01)
arr1 = standardize_along_axis(arr, axis=1)
assert np.allclose(arr1, np.array([[1.224, 0., -1.224], [-1.388, 0.462, 0.925]]), atol=0.01)
print("Tests were successful.")


## **Exercise 6**

You are given two time series stored in the arrays `temps` and `rain`, containing monthly mean temperatures and monthly precipitation totals. The two arrays have the same shape, `(n_year, 12)`, where `n_year` is the number of years covered by the time series, and 12 is the number of months in a year (January to December).
Solve the tasks below!

Example:
```
temps = np.array([[1.4, 3.4, 7.7, 13.3, 17.7, 21.4, 23.3, 23.2, 18.0, 12.7, 7.2, 2.2],
                  [0.7, 5.0, 8.1, 14.9, 17.5, 21.1, 23.1, 24.0, 18.9, 11.8, 7.0, 1.7]])
rain = np.array([[5, 12, 21, 54, 22, 85, 27, 46, 76, 10, 56, 62],
                 [91, 11, 34, 60, 112, 66, 73, 97, 23, 59, 93, 102]])
```

**A)** Define the `mean_yearly_rainfall_in_summer(rain)` function, which returns the total precipitation over the summer months (June to August), averaged over years.

Example:
```
mean_yearly_rainfall_in_summer(rain) -> 197.0
```

**B)** Define the `rain_in_coldest_month(temps, rain)` function, which returns the precipitation amount in the coldest month of the time series.

Example:
```
rain_in_coldest_month(temps, rain) -> 91.0
```

**C)** Define the `top3_highest_temps_each_year(temps)` function, which returns an array of shape `(n_year, 3)` containing, for each year, the three highest temperature values. Within each year, the returned values should be sorted in descending order.

Example:
```
top3_highest_temps_each_year(temps) -> np.array([[23.3, 23.2, 21.4], [24.0, 23.1, 21.1]])
```

**D)** Define the `temps_sorted_by_rainfall(temps, rain)` function, which returns a 1D array of shape `(n_year*12,)` containing the monthly temperatures sorted by the precipitation amount in the corresponding months, in descending order.

Example:
```
temps_sorted_by_rainfall(temps, rain) -> np.array([17.5, 1.7, 24., 7., 0.7, 21.4, 18., 23.1, 21.1, 2.2, 14.9, \
                                                   11.8, 7.2, 13.3, 23.2, 8.1, 23.3, 18.9, 17.7, 7.7, 3.4, 5., 12.7, 1.4])
```

**E)** Define the `coldest_month_of_rainiest_year(temps, rain)` function, which returns the index of the coldest month of the rainiest year as a 2-tuple: `(year_index, month_index)`.

Example:
```
coldest_month_of_rainiest_year(temps, rain) -> (1, 0)
```

**F)** Define the `rainy_months(rain)` function, which returns the indices of months that are rainier than the overall average (averaged over all months) as an array of shape `(m, 2)`, where each row is a `(year_index, month_index)` pair.

Example:
```
rainy_months(rain) -> np.array([[0, 5], [0, 8], [0, 10], [0, 11], [1, 0], [1, 3], \
                                [1,  4], [1, 5], [1, 6], [1, 7], [1, 9], [1, 10], [1, 11]])
```

**G)** Define the `rainy_months_of_the_year(rain)` function, which returns, in a 1D array, the month indices for which every year measured above-that-year-average precipitation.

Example:
```
rainy_months_of_the_year(rain) -> np.array([[7, 10, 11]])
```

**H)** Define the `driest_3months(rain)` function, which returns the index of the first month of the three-month period with the smallest total precipitation, as a 2-tuple: `(year_index, month_index)`.

Example:
```
driest_3months(rain) -> (0, 0)
```


### Hints

**Possible approaches:**

**Part A**
- Basic indexing, `np.sum` along an axis, then `np.mean`. **Difficulty:** 1/5

**Part B**
- Use `np.argmin`, then index into the flattened array (`np.reshape`). **Difficulty:** 2/5
- Use `np.argmin`, then convert the flat index to a 2D index with `np.unravel_index` and index the original array. **Difficulty:** 2/5

**Part C**
- `np.sort` along an axis, then basic indexing. **Difficulty:** 1/5

**Part D**
- `np.argsort` on the flattened array, then use the reversed index array to index the flattened other array. **Difficulty:** 2/5

**Part E**
- `np.sum` along an axis, `np.argmax`, `np.argmin`. **Difficulty:** 1/5

**Part F**
- `np.argwhere`, `>`, `np.mean`. **Difficulty:** 1/5

**Part G**
- `np.mean` along an axis, `>`, `np.all`, `np.where`. **Difficulty:** 2/5

**Part H**
- Build an index array that lists, in a 2D array, the month indices of all possible three-month periods, then use advanced indexing to extract the corresponding precipitation values. **Difficulty:** 3/5
- Using cumulative sums (`np.cumsum`): by subtracting a shifted version of the cumulative sum from itself, we obtain sums of consecutive length-`n` segments. Then use `np.argmin`/`np.argmax` and `np.unravel_index`. **Difficulty:** 2/5
- `np.lib.stride_tricks.sliding_window_view`. **Difficulty:** 2/5
- `np.lib.stride_tricks.as_strided`. **Difficulty:** 3/5


### Solutions

**Part A, solution 1**

In [ ]:
def mean_yearly_rainfall_in_summer(rain):
  return np.mean(np.sum(rain[:,5:8], axis=1))

**Part B, solution 1**

In [ ]:
def rain_in_coldest_month(temps, rain):
  coldest_month_idx = np.argmin(temps)   # flat idx (indexing flattened array)
  return rain.reshape(-1)[coldest_month_idx]

**Part B, solution 2**

In [ ]:
def rain_in_coldest_month(temps, rain):
  coldest_month_flat_idx = np.argmin(temps)   # flat idx (indexing flattened array)
  year_idx, month_in_year_idx = np.unravel_index(coldest_month_flat_idx, rain.shape)   # flat idx -> multi-dim index indexing original array
  return rain[year_idx, month_in_year_idx]

**Part C, solution 1**

In [ ]:
def top3_highest_temps_each_year(temps):
  return np.sort(temps, axis=1)[:,::-1][:,:3]

**Part D, solution 1**

In [ ]:
def temps_sorted_by_rainfall(temps, rain):
  sorter = np.argsort(rain.reshape(-1))[::-1]       # argsort with axis=None is an alternative to reshape
  return temps.reshape(-1)[sorter]

**Part E, solution 1**

In [ ]:
def coldest_month_of_rainiest_year(temps, rain):
  year_idx = np.argmax(np.sum(rain, axis=1))
  month_idx = np.argmin(temps[year_idx, :])
  return (year_idx, month_idx)

**Part F, solution 1**

In [ ]:
def rainy_months(rain):
  return np.argwhere(rain > np.mean(rain))


**Part G, solution 1**

In [ ]:
def rainy_months_of_the_year(rain):
  return np.where(np.all(rain > np.mean(rain, axis=1)[:,None], axis=0))[0]    # np.where returns a tuple of index arrays, since we are in 1D, [0] simply unpacks the tuple
                                                                              # an alternative would be: np.argwhere(...).T

**Part H, solution 1**

In [ ]:
def driest_3months(rain):
  idxs_ar = np.arange(rain.size-2)
  idxs = idxs_ar[:,None] + [0,1,2]     # broadcasting to (idxs_ar.shape[0], 3)
  rain_3months = rain.reshape(-1)[idxs]
  flat_min_idx = np.argmin(np.sum(rain_3months, axis=1))
  return np.unravel_index(flat_min_idx, rain.shape)

**Part H, solution 2**

In [ ]:
def driest_3months(rain):
  cumsum_rain = np.cumsum(rain)
  sum_3months = cumsum_rain[3:] - cumsum_rain[:-3]
  flat_min_idx = np.argmin(sum_3months)
  return np.unravel_index(flat_min_idx, rain.shape)

**Part H, solution 3**

In [ ]:
def driest_3months(rain):
  rain_3months = np.lib.stride_tricks.sliding_window_view(rain.reshape(-1), window_shape=(3,))
  flat_min_idx = np.argmin(np.sum(rain_3months, axis=1))
  return np.unravel_index(flat_min_idx, rain.shape)

**Part H, solution 4**

In [ ]:
def driest_3months(rain):
  rain_1d = rain.reshape(-1)         # explicit reshaping to force a contiguous memory layout
  sliding_window_shape = (rain_1d.shape[0]-2, 3)                          # (number of possible locations, length of sliding window)
  sliding_window_strides = (rain_1d.strides[0], rain_1d.strides[0])     # strides: number of bytes to move in memory to increase index along the different axes
  rain_3months = np.lib.stride_tricks.as_strided(rain_1d, shape=sliding_window_shape, strides=sliding_window_strides, writeable=False)   # use this function with caution (and possibly with the writable=False argument), see documentation
  flat_min_idx = np.argmin(np.sum(rain_3months, axis=1))
  return np.unravel_index(flat_min_idx, rain.shape)

**Tests**

In [ ]:
# Tests
temps = np.array([[1.4, 3.4, 7.7, 13.3, 17.7, 21.4, 23.3, 23.2, 18.0, 12.7, 7.2, 2.2],
                  [0.7, 5.0, 8.1, 14.9, 17.5, 21.1, 23.1, 24.0, 18.9, 11.8, 7.0, 1.7]], dtype=np.float32)   # (2, 12)
rain = np.array([[5, 12, 21, 54, 22, 85, 27, 46, 76, 10, 56, 62],
                 [91, 11, 34, 60, 112, 66, 73, 97, 23, 59, 93, 102]], dtype=np.float32)   # (2, 12)

if 'mean_yearly_rainfall_in_summer' in locals():
  ret = mean_yearly_rainfall_in_summer(rain)
  assert np.allclose(ret, 197., atol=0.01)
  print("Tests were successful for: 'mean_yearly_rainfall_in_summer'.")
else:
  print("'mean_yearly_rainfall_in_summer' not found, tests skipped.")

if 'rain_in_coldest_month' in locals():
  ret = rain_in_coldest_month(temps, rain)
  assert np.allclose(ret, 91., atol=0.01)
  print("Tests were successful for: 'rain_in_coldest_month'.")
else:
  print("'rain_in_coldest_month' not found, tests skipped.")

if 'top3_highest_temps_each_year' in locals():
  ret = top3_highest_temps_each_year(temps)
  assert np.allclose(ret, [[23.3, 23.2, 21.4], [24.0, 23.1, 21.1]], atol=0.01)
  print("Tests were successful for: 'top3_highest_temps_each_year'.")
else:
  print("'top3_highest_temps_each_year' not found, tests skipped.")

if 'temps_sorted_by_rainfall' in locals():
  ret = temps_sorted_by_rainfall(temps, rain)
  assert np.allclose(ret, [17.5, 1.7, 24., 7., 0.7, 21.4, 18., 23.1, 21.1, 2.2, 14.9, 11.8, 7.2, 13.3,\
                      23.2, 8.1, 23.3, 18.9, 17.7, 7.7, 3.4, 5., 12.7, 1.4], atol=0.01)
  print("Tests were successful for: 'temps_sorted_by_rainfall'.")
else:
  print("'temps_sorted_by_rainfall' not found, tests skipped.")

if 'coldest_month_of_rainiest_year' in locals():
  ret = coldest_month_of_rainiest_year(temps, rain)
  assert type(ret) is tuple
  assert np.array_equal(ret, (1, 0))
  print("Tests were successful for: 'coldest_month_of_rainiest_year'.")
else:
  print("'coldest_month_of_rainiest_year' not found, tests skipped.")

if 'rainy_months' in locals():
  ret = rainy_months(rain)
  ret_sorter1 = np.argsort(ret[:,1])
  ret = ret[ret_sorter1,:]
  ret_sorter0 = np.argsort(ret[:,0], kind='stable')
  ret = ret[ret_sorter0,:]
  assert np.array_equal(ret, [[0, 5], [0, 8], [0, 10], [0, 11], [1, 0], [1, 3], [1,  4], [1, 5], [1, 6], [1, 7], [1, 9], [1, 10], [1, 11]])
  print("Tests were successful for: 'rainy_months'.")
else:
  print("'rainy_months' not found, tests skipped.")

if 'rainy_months_of_the_year' in locals():
  ret = rainy_months_of_the_year(rain)
  assert np.array_equal(np.sort(ret), [7, 10, 11])
  print("Tests were successful for: 'rainy_months_of_the_year'.")
else:
  print("'rainy_months_of_the_year' not found, tests skipped.")

if 'driest_3months' in locals():
  ret = driest_3months(rain)
  assert type(ret) is tuple
  assert np.array_equal(ret, (0, 0))
  print("Tests were successful for: 'driest_3months'.")
else:
  print("'driest_3months' not found, tests skipped.")


## **Exercise 7**

Define the `index_of_1d(arr, vals_to_find, not_found_value)` function, which returns the indices of the elements of `vals_to_find` within the 1D array `arr`. The function returns an array with the same shape as `vals_to_find`. If a value occurs multiple times in `arr`, return the index of its first occurrence. If a value is not found in `arr`, place `not_found_value` at the corresponding position in the output. You can assume `arr` is 1D, but `vals_to_find` may have any shape.

Example:
```
arr = np.array([1, 5, 4, 5, 3, 6])
vals_to_find = np.array([[4, 2], [5, 6]])
index_of_1d(arr, vals_to_find, not_found_value=arr.shape[0]) -> ndarray([[2, 6], [1, 5]])
```


### Hints

**Possible approaches:**
- Compare the two arrays elementwise using "multiplication-table" style broadcasting, then extract matches and identify missing elements. **Difficulty:** 2/5
- `np.searchsorted`, `np.isin`. The challenge is that searching is done on sorted arrays, but we need indices in the original order (use `np.argsort` tricks). **Difficulty:** 3/5
- `np.searchsorted`, `np.isin`, but without `np.sort`/`np.argsort`. `np.unique` can sort the values and optionally return the indices of first occurrences in the original array, which we can exploit. **Difficulty:** 3/5


### Solutions

**Solution 1** - $~O(m\cdot n)$

In [ ]:
def index_of_1d(arr, vals_to_find, not_found_value):
  matches_mask = arr == vals_to_find.reshape(-1, 1)   # (n_vals_to_find, arr.size); alternative: vals_to_find.reshape(-1)[:,None]
  match_idxs = np.argmax(matches_mask, axis=1)        # (n_vals_to_find,)
  not_found_mask = ~np.any(matches_mask, axis=1)      # (n_vals_to_find,)
  match_idxs[not_found_mask] = not_found_value
  return match_idxs.reshape(vals_to_find.shape)

**Solution 2** - $~O((m + n)\cdot log\ n)$

In [ ]:
def index_of_1d(arr, vals_to_find, not_found_value):
  sorter = np.argsort(arr)
  match_idxs = np.searchsorted(arr, vals_to_find, sorter=sorter)   # instead of sorting 'arr', we specify the sorter
  not_found_mask = ~np.isin(vals_to_find, arr)                     # vals_to_find.shape
  match_idxs[match_idxs >= arr.shape[0]] = 0                       # np.searchsorted() can return values == arr.shape[0] if not found and higher than maximum of arr
                                                                   #   this would result in an out of bounds exception when indexing the sorter
                                                                   #   these are values that were not found so their idx will be overwritten later anyway
  match_idxs = sorter[match_idxs]                                  # using the sorter to transform indices back to the original 'arr' order
                                                                   #   sorter[idxs_into_sorted_array] gives back idxs into unsorted array
  match_idxs[not_found_mask] = not_found_value
  return match_idxs

**Solution 3** - $~O((m + n)\cdot log\ n)$

In [ ]:
def index_of_1d(arr, vals_to_find, not_found_value):
  u_arr, ind_arr = np.unique(arr, return_index=True)
  not_found_mask = ~np.isin(vals_to_find, u_arr)                     # vals_to_find.shape
  u_match_idxs = np.searchsorted(u_arr, vals_to_find)
  u_match_idxs[u_match_idxs >= u_arr.shape[0]] = 0                   # same trick as in the previous solution to avoid out of bounds exception
                                                                     #    when indexing ind_arr
  match_idxs = ind_arr[u_match_idxs]
  match_idxs[not_found_mask] = not_found_value
  return match_idxs

**Tests**

In [ ]:
# Tests
arr = np.array([1, 5, 10, 5, 3, 6])
vals_to_find = np.array([[10, 2], [5, 6], [6, 15]])
ret = index_of_1d(arr, vals_to_find, not_found_value=arr.shape[0])
assert np.array_equal(ret, np.array([[2, 6], [1, 5], [5, 6]]))
print("Tests were successful.")

## **Exercise 8**

Write the `triangle_area_perimeter(edges)` function, where `edges` is an array of shape `(n_triples, 3)`. The function returns the indices (along axis #0) of those triples in `edges` that can form a triangle when interpreted as side lengths, as a 1D array of shape `(n_valid_triples,)`. In addition to the indices, the function should also return the perimeter and area of the corresponding triangles in an array of shape `(n_valid_triples, 2)`.

For a triple to represent valid triangle side lengths, the triangle inequality must hold and every side length must be positive. The triangle area can be computed from the side lengths using Heron's formula.

Example:
```
edges = np.array([[1., 2., -2.],
                  [5., 3., 4.],
                  [4., 2., 3.]])
valid_triangle_idxs, area_perimeter = triangle_area_perimeter(edges)

valid_triangle_idxs -> [1, 2]
area_perimeter -> [[6., 12.], [2.904, 9.]]
```


### Hints

**Possible approaches:**
- Apply the triangle inequality with logical operations, then compute perimeter and Heron's formula in a vectorized way. **Difficulty:** 2/5


### Solutions

**Solution 1**

In [ ]:
def triangle_area_perimeter(edges):
  valid_tr_mask = (edges[:,0] + edges[:,1] > edges[:,2]) & (edges[:,0] + edges[:,2] > edges[:,1]) &\
                  (edges[:,1] + edges[:,2] > edges[:,0]) & np.all(edges >= 0., axis=1)  # shape: (n_triples, ) bool mask
  valid_triangle_idxs = np.where(valid_tr_mask)[0]  # shape: (n_valid_triangles, ) integer idx array
  edges = edges[valid_triangle_idxs]    # shape: (n_valid_triangles, 3) float array
  perimeter = np.sum(edges, axis=1)     # shape: (n_valid_triangles,) float array
  semi_per = perimeter*.5
  area = np.sqrt(semi_per*(semi_per-edges[:,0])*(semi_per-edges[:,1])*(semi_per-edges[:,2]))      # shape: (n_valid_triangles,) float array
  area_perimeter = np.stack([area, perimeter], axis=1)     # shape: (n_valid_triangles, 2) float array
  return valid_triangle_idxs, area_perimeter

**Tests**

In [ ]:
# Tests
edges = np.array([[1., 2., 0.], [0., 2., 1.], [2.3, -2., -5.5], [0., 0., 0.],\
                  [4., 2., 3.], [2., 3.01, 1.],  [1., 1., 1.], [1., 1.99, 1.]])
valid_triangle_idxs, area_perimeter = triangle_area_perimeter(edges)
assert valid_triangle_idxs.shape == (3,)
assert np.all(valid_triangle_idxs == np.array([4,6,7]))
assert area_perimeter.shape == (3,2)
assert np.allclose(area_perimeter, np.array([[2.90473751, 9.], [0.4330127, 3.], [0.09937555, 3.99]]))
print("Tests were successful.")

## **Exercise 9**

Define the `local_extrema_idxs_1d(arr)` function, which takes a 1D array with at least 3 elements. Treating `arr` as a sequence, return the indices of those elements that are strict local extrema. Strict local extrema are elements that are strictly smaller than both neighbors or strictly larger than both neighbors, and have neighbors on both sides.

Example:
```
arr = np.array([1.2, 3.4, 2.5, 5.5, 6.7, 7.7, 7.7, 6.5, 5.2, 3.1, 4.4, 2.2, 1.1, 0.1])
local_extrema_idxs_1d(arr) -> ndarray([1, 2, 9, 10])
```


### Hints

**Possible approaches:**
- Compare array slices shifted by 1 relative to each other, then combine with logical operations. **Difficulty:** 2/5
- Similar to the previous one, but shorter using `np.sign`. **Difficulty:** 2/5
- `np.lib.stride_tricks.sliding_window_view` with length-3 windows: the middle element of each window is either smaller or larger than the two neighbors. **Difficulty:** 2/5
- Apply `np.sign` to the sequence of differences between neighboring elements. The product of neighboring elements of this sign sequence is exactly -1 where there is a local extremum. **Difficulty:** 2/5


### Solutions

**Solution 1**

In [ ]:
def local_extrema_idxs_1d(arr):
  local_maxima = (arr[1:-1] > arr[2:]) & (arr[1:-1] > arr[:-2])    # (arr.shape[0]-2,) bool
  local_minima = (arr[1:-1] < arr[2:]) & (arr[1:-1] < arr[:-2])    # (arr.shape[0]-2,) bool
  local_extrema = local_minima | local_maxima
  local_extrema_idxs = np.where(local_extrema)[0] + 1
  return local_extrema_idxs

**Solution 2**

In [ ]:
def local_extrema_idxs_1d(arr):
  sign_sum = np.sign(arr[1:-1] - arr[2:]) + np.sign(arr[1:-1] - arr[:-2])    # (arr.shape[0]-2,) int
  local_minima_idxs, = np.where(sign_sum == -2)
  local_maxima_idxs, = np.where(sign_sum == 2)
  local_extrema_idxs = np.union1d(local_minima_idxs, local_maxima_idxs) + 1  # this is an alternative to the elementwise OR in solution#1
  return local_extrema_idxs

**Solution 3**

In [ ]:
def local_extrema_idxs_1d(arr):
  arr_sliding = np.lib.stride_tricks.sliding_window_view(arr, window_shape=(3,))   # (arr.shape[0]-2, 3)
  local_maxima = (arr_sliding[:,1] > arr_sliding[:,2]) & (arr_sliding[:,1] > arr_sliding[:,0])    # (arr.shape[0]-2,) bool
  local_minima = (arr_sliding[:,1] < arr_sliding[:,2]) & (arr_sliding[:,1] < arr_sliding[:,0])    # (arr.shape[0]-2,) bool
  local_extrema_idxs, = np.where(local_minima | local_maxima)
  return local_extrema_idxs + 1

**Solution 4**

In [ ]:
def local_extrema_idxs_1d(arr):
  arr_diff_sgn = np.sign(np.diff(arr))
  local_extrema_idxs, = np.where(arr_diff_sgn[1:] * arr_diff_sgn[:-1] == -1)
  return local_extrema_idxs + 1

**Tests**

In [ ]:
# Tests
arr = np.array([1.2, 3.4, 2.5, 5.5, 6.7, 7.7, 7.7, 6.5, 5.2, 3.1, 4.4, 2.2, 1.1, 0.1])
ret = local_extrema_idxs_1d(arr)
assert np.array_equal(np.sort(ret), np.array([1, 2, 9, 10]))
print("Tests were successful.")